# VWAP Events 04 — fading extreme displacement from VWAP

Protocol: `discussions/VWAP_Events/006_claude_ve04_preregistration.md` (fixed before this run).

Displacement ≥ 3/4/5σ·√15 or ≥ 0.5/0.75/1% from VWAP (any time, 30-min cooldown) → fade toward VWAP → target / VWAP touch / stop / 60–240 min.
**Cell chosen on DEV (before 2026-04-01), judged once on TEST.** G1 hour-matched real − placebo > 0; G2 TEST gross lower bound > 7 bp.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import vwap_events as ve, vwap_fade as vf, vwap_extreme as vx
importlib.reload(vx)
SYNTHETIC=os.getenv('VE04_SYNTHETIC','0')=='1';ASSETS=['BTC','ETH'];xp=vx.XParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='ve04-');OUT=Path(tmp.name)/'04';xp=vx.XParams(n_boot=300)
else:
    OUT=ROOT/'notebooks'/'VWAP_Events'/'04'
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_vwap_extreme.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-300:]);assert chk.returncode==0,'tests failed'
print(json.dumps(vx.registration(xp),indent=1,default=str))

In [ ]:
T0=time.time();parts=[]
for i,a in enumerate(ASSETS):
    df=vf.synthetic_path_klines(n_days=200,seed=4+i,start='2025-10-01') if SYNTHETIC else pd.read_parquet(BIN/f'{a}USDT_um_1m.parquet',columns=['open_time','high','low','close','volume','quote_volume'])
    parts.append(vx.run_asset(df,a,xp));print(a,f'{time.time()-T0:.0f}s')
R=pd.concat(parts,ignore_index=True);print('simulated trades:',len(R))
cnt=R[(R.target=='vwap')&(R.stop==R.stop.min())&(R.hold==R.hold.min())].groupby(['kind','thr','test'],observed=True).size().unstack('test')
print('events per threshold (real / placebo; DEV=False, TEST=True):');display(cnt)

## Selection on DEV, decision on TEST

In [ ]:
dev=vx.dev_table(R,xp);dev.to_csv(OUT/'04_dev_selection.csv',index=False)
display(dev.sort_values('dev_lo',ascending=False).head(10).round(2))
cell=vx.select(dev,xp);print('selected cell:',cell)
res=vx.evaluate(R,cell,xp);print(json.dumps(res,indent=1,default=float))

## All cells (descriptive)

In [ ]:
FT=vx.full_table(R,xp);FT.to_csv(OUT/'04_all_cells.csv',index=False)
for test in (False,True):
    print('TEST' if test else 'DEV','— real gross bp (rows thr x hold, cols target x stop)')
    display(FT[(FT.kind=='real')&(FT.test==test)].pivot_table(index=['thr','hold'],columns=['target','stop'],values='gross',observed=True).round(1))
print('TEST real - placebo, VWAP-touch exit:')
v=FT[(FT.target=='vwap')&(FT.test)].pivot_table(index=['thr','hold','stop'],columns='kind',values=['gross','win','n'],observed=True)
display(v.round(2))

In [ ]:
if not SYNTHETIC:
    (OUT/'04_decision.json').write_text(json.dumps(res,indent=1,default=float))
    (OUT/'04_manifest.json').write_text(json.dumps({'registration':vx.registration(xp),'runtime_s':time.time()-T0,'trades':int(len(R))},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')